In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import plotly.graph_objects as go

from data.fetcher import OptionChainFetcher
from data.cleaner import clean_option_chain
from models.surface import IVSurface
from analytics.geometry import compute_all_geometry_metrics
from analytics.regime_classifier import RegimeClassifier, MarketRegime
from visualization.regime_visuals import (
    plot_regime_gauge,
    plot_regime_probabilities,
    plot_early_warning_dashboard
)

In [ ]:
# Fetch and process data
fetcher = OptionChainFetcher('SPY')
chain = fetcher.fetch_current_chain()
clean_chain = clean_option_chain(chain)
surface = IVSurface(clean_chain)

# Compute metrics
metrics = compute_all_geometry_metrics(surface)
atm_vol = surface.get_atm_vol(30)

In [ ]:
# Initialize classifier
classifier = RegimeClassifier()

# Classify current regime
result = classifier.classify(
    atm_vol=atm_vol,
    skew=metrics.get('skew_25d', 0),
    curvature=metrics.get('curvature_atm', 0),
    butterfly=metrics.get('butterfly_atm', 0)
)

print("Regime Classification Results:")
print(f"  Current Regime: {result['current_regime']}")
print(f"  Confidence: {result['confidence']:.1%}")
print(f"  Key Drivers: {', '.join(result['key_drivers'])}")

In [ ]:
# Regime probabilities
print("\nRegime Probabilities:")
for regime, prob in result['regime_probability'].items():
    print(f"  {regime}: {prob:.1%}")

In [ ]:
# Visualize regime gauge
fig = plot_regime_gauge(result)
fig.show()

In [ ]:
# Visualize probabilities
fig = plot_regime_probabilities(result['regime_probability'])
fig.show()

In [ ]:
# Early warning dashboard
warning_data = {
    'percentiles': {
        'atm_vol': result.get('atm_percentile', 50),
        'curvature': result.get('curvature_percentile', 50),
        'skew': result.get('skew_percentile', 50)
    }
}

fig = plot_early_warning_dashboard(warning_data)
fig.show()